In [1]:
library(tidyverse)
library(tis)
library(baseballr)
library(hoopR)
library(MASS)
library(glmnet)
library(mpath)
library(glmmTMB)
library(Matrix)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘tis’


The following objects are masked from ‘package:lubridate’:

    day, hms, month, period, POSIXct, quarter, today, year, ymd


The following object is masked from ‘package:dplyr’:

    between



Attaching package: ‘MASS’


The following object is masked from ‘package:dplyr’:

    select


Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glm

In [2]:
setwd('..')
getwd()

#source('./code/filter-and-add-features.r')

[1] "/accounts/masters/ben_khothsombath/repos/230A-final-project"

In [3]:
df <- read_csv("./data/final-dataset.csv")
dim(df)

Rows: 1130841 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (3): destination, day, season
dbl  (2): hour, ridership
lgl  (9): destination_green, destination_red, destination_yellow, destinatio...
date (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] 1130841      15

In [4]:
head(df)

date,hour,destination,ridership,day,destination_green,destination_red,destination_yellow,destination_blue,destination_orange,is_holiday,is_giants_home,is_as_home,warriors_at_chase,season
<date>,<dbl>,<chr>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>
2023-01-01,0,12TH,63,sunday,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,winter
2023-01-01,0,16TH,78,sunday,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,winter
2023-01-01,0,19TH,27,sunday,FALSE,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,winter
2023-01-01,0,24TH,72,sunday,TRUE,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,winter
2023-01-01,0,ANTC,13,sunday,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,winter
2023-01-01,0,ASHB,15,sunday,FALSE,TRUE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,winter


In [5]:
print(format(object.size(df), units = "GB"))

[1] "0.1 Gb"


In [6]:
# Split into Training and Testing sets

train <- df %>% filter(year(date) < 2025)
test <- df %>% filter(year(date) == 2025)

# Treat hour as a factor
train <- train %>% mutate(hour = as_factor(hour))
test <- test %>% mutate(hour = as_factor(hour))

print(dim(train))
print(dim(test))

[1] 754747     15
[1] 376094     15


In [7]:
basic_lm <- lm(ridership~., data=train[, -c(1)])
summary(basic_lm)
#plot(basic_lm)


Call:
lm(formula = ridership ~ ., data = train[, -c(1)])

Residuals:
   Min     1Q Median     3Q    Max 
-643.4  -55.4   -6.1   40.6 5212.9 

Coefficients: (5 not defined because of singularities)
                        Estimate Std. Error t value Pr(>|t|)    
(Intercept)              40.8235     1.9799  20.619  < 2e-16 ***
hour1                   -12.0001     1.4884  -8.062 7.51e-16 ***
hour2                    21.3255     3.5417   6.021 1.73e-09 ***
hour3                  -238.8812     6.2557 -38.186  < 2e-16 ***
hour4                   -91.2979     2.2668 -40.276  < 2e-16 ***
hour5                    -8.2757     1.5481  -5.346 9.01e-08 ***
hour6                    72.0489     1.4835  48.568  < 2e-16 ***
hour7                   168.6822     1.4360 117.467  < 2e-16 ***
hour8                   245.8035     1.4183 173.310  < 2e-16 ***
hour9                   177.4794     1.4157 125.361  < 2e-16 ***
hour10                  109.9593     1.4157  77.670  < 2e-16 ***
hour11                

In [8]:
# Log-transformed lm
log_lm <- lm(log(ridership) ~ ., data=train[, -c(1)])
summary(log_lm)
#plot(log_lm)


Call:
lm(formula = log(ridership) ~ ., data = train[, -c(1)])

Residuals:
    Min      1Q  Median      3Q     Max 
-5.6802 -0.3228  0.0037  0.3494  5.0606 

Coefficients: (5 not defined because of singularities)
                        Estimate Std. Error  t value Pr(>|t|)    
(Intercept)             3.164718   0.006921  457.243  < 2e-16 ***
hour1                  -1.469782   0.005203 -282.470  < 2e-16 ***
hour2                  -1.714138   0.012381 -138.447  < 2e-16 ***
hour3                  -3.226646   0.021869 -147.546  < 2e-16 ***
hour4                  -2.489483   0.007924 -314.154  < 2e-16 ***
hour5                  -0.433319   0.005412  -80.068  < 2e-16 ***
hour6                   0.912438   0.005186  175.946  < 2e-16 ***
hour7                   1.377228   0.005020  274.350  < 2e-16 ***
hour8                   1.773581   0.004958  357.716  < 2e-16 ***
hour9                   1.634921   0.004949  330.342  < 2e-16 ***
hour10                  1.459760   0.004949  294.954  < 2e-16

In [9]:
poisson_lm <- glm(ridership ~ ., data = train[, -c(1)], family = poisson(link = "log"))
summary(poisson_lm)


Call:
glm(formula = ridership ~ ., family = poisson(link = "log"), 
    data = train[, -c(1)])

Coefficients: (5 not defined because of singularities)
                         Estimate Std. Error  z value Pr(>|z|)    
(Intercept)             3.0554494  0.0013326  2292.76   <2e-16 ***
hour1                  -1.3831462  0.0027807  -497.42   <2e-16 ***
hour2                  -1.5549157  0.0098150  -158.42   <2e-16 ***
hour3                  -3.4075701  0.0226773  -150.26   <2e-16 ***
hour4                  -2.4834039  0.0064054  -387.70   <2e-16 ***
hour5                   0.4111331  0.0015202   270.44   <2e-16 ***
hour6                   1.5269704  0.0012679  1204.30   <2e-16 ***
hour7                   2.1879591  0.0012040  1817.28   <2e-16 ***
hour8                   2.5322328  0.0011851  2136.76   <2e-16 ***
hour9                   2.2371284  0.0011998  1864.52   <2e-16 ***
hour10                  1.8193425  0.0012294  1479.87   <2e-16 ***
hour11                  1.7134905  0.0012389

In [10]:
#plot(poisson_lm)

In [11]:
model_nb <- glm.nb(ridership ~ ., data = train[, -c(1)])
summary(model_nb)


Call:
glm.nb(formula = ridership ~ ., data = train[, -c(1)], init.theta = 3.1850146, 
    link = log)

Coefficients: (5 not defined because of singularities)
                        Estimate Std. Error  z value Pr(>|z|)    
(Intercept)             3.572892   0.006031  592.418   <2e-16 ***
hour1                  -1.349761   0.005195 -259.840   <2e-16 ***
hour2                  -1.541870   0.014219 -108.435   <2e-16 ***
hour3                  -3.541680   0.028867 -122.690   <2e-16 ***
hour4                  -2.619377   0.009536 -274.695   <2e-16 ***
hour5                  -0.311511   0.004959  -62.811   <2e-16 ***
hour6                   0.869317   0.004607  188.710   <2e-16 ***
hour7                   1.450069   0.004438  326.743   <2e-16 ***
hour8                   1.701822   0.004379  388.628   <2e-16 ***
hour9                   1.466418   0.004381  334.730   <2e-16 ***
hour10                  1.232497   0.004392  280.641   <2e-16 ***
hour11                  1.232990   0.004392  280.

In [23]:
# Remove the intercept since glmnet adds it automatically
model_formula <- as.formula("ridership ~ hour + destination + day + is_holiday + is_giants_home + is_as_home + warriors_at_chase + season - 1")

X_train <- model.matrix(model_formula, data = train)
y_train <- train$ridership

X_test <- model.matrix(model_formula, data = test)

# Time-based split for CV
n <- nrow(train)
fold_size <- floor(n / 5)

foldid <- rep(1:5, each = fold_size, length.out = n)

# Fit Ridge Poisson
cv_ridge_poisson <- cv.glmnet(
    x = X_train,
    y = y_train,
    family = "poisson",
    alpha = 0, # 0 for ridge, 1 for LASSO
    foldid = foldid
)

# Best lambda
best_lambda_poisson <- cv_ridge_poisson$lambda.min
best_lambda_poisson

[1] 7.081652

The best Lambda for the Poisson Regression was 7.08165174440212.

In [18]:
# Note - technically with a time-based split, you want to train sequentially (for example, you don't want to test block 4
# Using blocks 1, 2, 3, and 5 because 5 will be using future data to predict 4
# But we're just going to assume that future data is similar to present data and we'll discuss this
# And show ridership over time to see if there is a trend

# 5-fold CV for ridge hyperparameter selection uses way too much space
# Let's instead use a validation set

sub_train <- train %>% filter(year(date) == 2023)
sub_val <- train %>% filter(year(date) == 2024)

X_sub_train <- sparse.model.matrix(model_formula, data = sub_train)
X_sub_val <- sparse.model.matrix(model_formula, data = sub_val)

y_sub_train <- sub_train$ridership
y_sub_val <- sub_val$ridership

X_test <- sparse.model.matrix(model_formula, data = test)

# First, estimate theta from a small, unregularized fit
nb_for_theta <- glm.nb(
  ridership ~ hour + destination + day + is_holiday +
              is_giants_home + is_as_home + warriors_at_chase + season,
  data = sub_train
)
theta_est <- nb_for_theta$theta
cat("Estimated theta:", theta_est, "\n")

lambdas <- 10^seq(2, -3, length.out = 10)
mse_results <- numeric(length(lambdas))

# Loop through lambdas
for (i in seq_along(lambdas)) {
    fit <- glmnet(
    x = X_sub_train,
    y = y_sub_train,
    family = negative.binomial(theta = theta_est),
    alpha = 0,
    lambda = lambdas[i]
  )

    # Predict on 2024 data
    preds <- as.numeric(predict(fit, newx = X_sub_val, type = "response"))
    mse_results[i] <- mean((y_sub_val - preds)^2)
    message(sprintf("Lambda: %8.5f | MSE: %.2f", lambdas[i], mse_results[i]))
}

best_lambda_nb <- lambdas[which.min(mse_results)]
cat("Best lambda (Ridge NB):", best_lambda_nb, "\n")

Estimated theta: 3.222975 


Lambda: 100.00000 | MSE: 60552.98

Lambda: 27.82559 | MSE: 58015.55

Lambda:  7.74264 | MSE: 52617.15

Lambda:  2.15443 | MSE: 44532.85

Lambda:  0.59948 | MSE: 39237.19

Lambda:  0.16681 | MSE: 39975.03

Lambda:  0.04642 | MSE: 41641.89

Warning message:
“glmnet.fit: algorithm did not converge”
Lambda:  0.01292 | MSE: 42403.15

Lambda:  0.00359 | MSE: 42660.37

Lambda:  0.00100 | MSE: 42734.61



Best lambda (Ridge NB): 0.5994843 


In [20]:
# Re-estimate theta on full training set
nb_for_theta_full <- glm.nb(
    ridership ~ hour + destination + day + is_holiday + is_giants_home + is_as_home + warriors_at_chase + season,
    data = train
)

final_nb_ridge <- glmnet(
    x = X_train,
    y = y_train,
    family = negative.binomial(theta = nb_for_theta_full$theta),
    alpha = 0,
    lambda = best_lambda_nb
)

summary(final_nb_ridge)

          Length Class     Mode   
a0         1     -none-    numeric
beta      82     dgCMatrix S4     
df         1     -none-    numeric
dim        2     -none-    numeric
lambda     1     -none-    numeric
dev.ratio  1     -none-    numeric
nulldev    1     -none-    numeric
npasses    1     -none-    numeric
jerr       1     -none-    numeric
offset     1     -none-    logical
call       6     -none-    call   
family    12     family    list   
nobs       1     -none-    numeric

Now let's try a zero-inflated NB. I'm not sure if it's possible to do ridge + zero-inflated, but ridge might offer minimal improvement anyway since we're working with all categorical variables.

In [21]:
model_zinb <- glmmTMB(
    ridership ~ hour + destination + day + is_holiday + is_giants_home + is_as_home + warriors_at_chase + season,
    data = train,
    ziformula = ~1,
    family = nbinom2 # nbinom2 = NB with quadratic variance (same as glm.nb)
)

summary(model_zinb)

Warning message in finalizeTMB(TMBStruc, obj, fit, h, data.tmb.old):
“Model convergence problem; iteration limit reached without convergence (10). See vignette('troubleshooting'), help('diagnose')”


 Family: nbinom2  ( log )
Formula:          
ridership ~ hour + destination + day + is_holiday + is_giants_home +  
    is_as_home + warriors_at_chase + season
Zero inflation:             ~1
Data: train

     AIC      BIC   logLik deviance df.resid 
 7622627  7623596 -3811230  7622459   754663 


Dispersion parameter for nbinom2 family (): 3.19 

Conditional model:
                       Estimate Std. Error z value Pr(>|z|)    
(Intercept)            3.561882   0.006034   590.3  < 2e-16 ***
hour1                 -1.356092   0.005401  -251.1  < 2e-16 ***
hour2                 -1.534601   0.014928  -102.8  < 2e-16 ***
hour3                 -3.483621   0.031752  -109.7  < 2e-16 ***
hour4                 -2.629049   0.010045  -261.7  < 2e-16 ***
hour5                 -0.318064   0.005057   -62.9  < 2e-16 ***
hour6                  0.863237   0.004715   183.1  < 2e-16 ***
hour7                  1.443713   0.004548   317.4  < 2e-16 ***
hour8                  1.695506   0.004487   377.9  < 2e

In [22]:
nb_for_theta_full$theta

[1] 3.185015

The value of theta for the NB is 3.18501460012295